# Multiple Linear Regression

## Data Loading & Exploration

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn import metrics


def section(title):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)


# Read CSV
df = pd.read_csv("../Day 11/data/advertising.csv")

section("LAST 5 ROWS")
print(df.tail())

section("SHAPE")
print("Shape:", df.shape)

section("COLUMNS")
print("Columns:", df.columns.tolist())

section("DATA TYPES & INFO")
print(df.info())

section("MISSING VALUES")
print(df.isnull().sum())

section("STATISTICAL SUMMARY")
print(df.describe())

section("DUPLICATE ROWS")
print("Duplicate rows:", df.duplicated().sum())



LAST 5 ROWS
        TV  Radio  Newspaper  Sales
195   38.2    3.7       13.8    7.6
196   94.2    4.9        8.1   14.0
197  177.0    9.3        6.4   14.8
198  283.6   42.0       66.2   25.5
199  232.1    8.6        8.7   18.4

SHAPE
Shape: (200, 4)

COLUMNS
Columns: ['TV', 'Radio', 'Newspaper', 'Sales']

DATA TYPES & INFO
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   TV         200 non-null    float64
 1   Radio      200 non-null    float64
 2   Newspaper  200 non-null    float64
 3   Sales      200 non-null    float64
dtypes: float64(4)
memory usage: 6.4 KB
None

MISSING VALUES
TV           0
Radio        0
Newspaper    0
Sales        0
dtype: int64

STATISTICAL SUMMARY
               TV       Radio   Newspaper       Sales
count  200.000000  200.000000  200.000000  200.000000
mean   147.042500   23.264000   30.554000   15.130500
std     85.854236   14.

## Feature and Target Selection

In [2]:
X = df[["TV", "Radio", "Newspaper"]]
y = df["Sales"]

print(X.shape)
print(y.shape)


(200, 3)
(200,)


## Train-Test Split

In [3]:
X_train_bow, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set shape:", X_train_bow.shape, y_train.shape)
print("Testing set shape:", X_test.shape, y_test.shape)


Training set shape: (160, 3) (160,)
Testing set shape: (40, 3) (40,)


## Model Training

In [4]:
mr_model = LinearRegression()
mr_model.fit(X_train_bow, y_train)

print("Model coefficients:", mr_model.coef_)  # (Weights for TV, Radio, and Newspaper)
print("Model intercept:", mr_model.intercept_)  # Intercept of the regression line

Model coefficients: [0.05450927 0.10094536 0.00433665]
Model intercept: 4.714126402214127


## Overfitting Check (Train vs Test R²)

In [5]:
# Overfitting check: Train and Test Scores
train_score = mr_model.score(X_train_bow, y_train)
test_score = mr_model.score(X_test, y_test)

print(f"Training Score (R^2): {train_score:.4f}")
print(f"Testing Score (R^2): {test_score:.4f}")


Training Score (R^2): 0.9001
Testing Score (R^2): 0.9059


## RMSE Evaluation

In [6]:
# Train RMSE
train_predictions = mr_model.predict(X_train_bow)
train_rmse = np.sqrt(metrics.mean_squared_error(y_train, train_predictions))

# Test RMSE
test_predictions = mr_model.predict(X_test)
test_rmse = np.sqrt(metrics.mean_squared_error(y_test, test_predictions))

print(f"Train RMSE: {train_rmse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")


Train RMSE: 1.6359
Test RMSE: 1.7052


## MAE Evaluation

In [7]:
print(f"Train MAE: {metrics.mean_absolute_error(y_train, train_predictions):.4f}")
print(f"Test MAE: {metrics.mean_absolute_error(y_test, test_predictions):.4f}")

Train MAE: 1.2344
Test MAE: 1.2748


## R² Score Summary

In [8]:
print(f"Train R2 Score: {mr_model.score(X_train_bow, y_train):.4f}")
print(f"Test R2 Score: {mr_model.score(X_test, y_test):.4f}")

Train R2 Score: 0.9001
Test R2 Score: 0.9059


## Overfitting Analysis: RMSE, MAE, R² Comparison

**How to read RMSE / MAE / R2 to decide "overfitting or not":**

Overfitting = model memorizes training data instead of learning general patterns. Symptom: it does GREAT on train but MUCH WORSE on unseen test data. The check is always the same shape: compare each metric's train value vs its test value.

- **RMSE & MAE** (error metrics, lower is better):
  - If Test >> Train -> errors balloon on unseen data -> overfitting.
  - If they're close -> model errs by roughly the same amount on both sets -> good generalization.
- **R2** (goodness-of-fit, higher/closer to 1 is better):
  - If Train R2 >> Test R2 (big drop) -> model explains train variance well but fails on test -> overfitting.
  - If Train R2 ~= Test R2 (or test is even slightly higher) -> consistent performance -> no overfitting.
  - If both R2 scores are low -> that's UNDERFITTING instead (model too simple, not enough signal captured).

We use **R2** as the deciding metric below because it's on a fixed, interpretable scale (0 to 1), unlike RMSE/MAE which depend on the units of Sales. **0.05** is a rule-of-thumb threshold: a gap that small is normal randomness from the train/test split, not overfitting. The `"Gap (Test - Train)"` column is the key diagnostic: a large positive gap on RMSE/MAE, or a large negative gap on R2, signals overfitting.


In [9]:
comparison = pd.DataFrame(
    {
        "Train": [
            train_rmse,
            metrics.mean_absolute_error(y_train, train_predictions),
            train_score,
        ],
        "Test": [
            test_rmse,
            metrics.mean_absolute_error(y_test, test_predictions),
            test_score,
        ],
    },
    index=["RMSE", "MAE", "R2"],
)
comparison["Gap (Test - Train)"] = comparison["Test"] - comparison["Train"]

section("TRAIN VS TEST COMPARISON")
print(comparison)

r2_gap = train_score - test_score  # positive -> train did better than test
if r2_gap > 0.05:
    print(
        f"\nConclusion: R2 drops by {r2_gap:.4f} from train to test -> model is OVERFITTING."
    )
elif r2_gap < -0.05:
    print(
        f"\nConclusion: Test R2 is higher than train R2 by {abs(r2_gap):.4f} -> unusual, check the split/data."
    )
else:
    print(
        f"\nConclusion: RMSE, MAE, and R2 are close between train and test (R2 gap = {r2_gap:.4f}) -> model GENERALIZES WELL, no overfitting."
    )



TRAIN VS TEST COMPARISON
         Train      Test  Gap (Test - Train)
RMSE  1.635892  1.705215            0.069323
MAE   1.234416  1.274826            0.040410
R2    0.900142  0.905901            0.005760

Conclusion: RMSE, MAE, and R2 are close between train and test (R2 gap = -0.0058) -> model GENERALIZES WELL, no overfitting.


## Prediction on New Data

In [10]:
tv = 200
radio = 30
newspaper = 10

predictions_df = mr_model.predict(
    pd.DataFrame(
        {
            "TV": [tv],
            "Radio": [radio],
            "Newspaper": [newspaper],
        }
    )
)

print(
    f"Predicted Sales for TV={tv}, Radio={radio}, Newspaper={newspaper}: {predictions_df[0]:.4f}"
)

Predicted Sales for TV=200, Radio=30, Newspaper=10: 18.6877


---

# Logistic Regression

We now load the dataset for the classification part of this notebook: `"Review Text"` (raw customer review) as the feature, `"sentiment"` (-1/1) as the target — this switches the task from regression (predicting a number) to classification (predicting a category).

In [11]:
df_reviews = pd.read_csv("data/womens_clothing_ecommerce_reviews.csv")
print(df_reviews.head())

                                         Review Text  sentiment
0  Absolutely wonderful - silky and sexy and comf...          1
1  Love this dress!  it's sooo pretty.  i happene...          1
2  I love, love, love this jumpsuit. it's fun, fl...          1
3  This shirt is very flattering to all due to th...          1
4  I love tracy reese dresses, but this one is no...         -1


## Data Quality Check (Empty, Null, Duplicates)

### Data Types & Missing Values

Even though the text column is stored as a string dtype, a missing review can still show up as NaN — `isnull()` catches it, dtype alone does not.

In [12]:
section("DATA TYPES")
print("Dtype of each column:\n", df_reviews.dtypes)

section("MISSING (NULL) VALUES")
print(df_reviews.isnull().sum())


DATA TYPES
Dtype of each column:
 Review Text      str
sentiment      int64
dtype: object

MISSING (NULL) VALUES
Review Text    0
sentiment      0
dtype: int64


Both `"Review Text"` and `"sentiment"` have 0 nulls — no missing-value cleanup needed before modeling.

### Duplicate Rows

Duplicate reviews would let the same example influence training multiple times, effectively giving it more weight and biasing the model.

In [13]:
section("DUPLICATE ROWS")
print("Duplicate rows:", df_reviews.duplicated().sum())

df_reviews.drop_duplicates(inplace=True)
print("Duplicate rows after drop:", df_reviews.duplicated().sum())


DUPLICATE ROWS
Duplicate rows: 7
Duplicate rows after drop: 0


7 duplicate rows out of ~19.8k — a tiny fraction, but worth dropping (`drop_duplicates()`) before training to avoid any bias.

### Empty Review Column

`isnull()` alone would miss a review that's an empty/whitespace-only string instead of true NaN, so check both explicitly.

In [14]:
section("EMPTY REVIEWS")
empty_reviews_count = (
    df_reviews["Review Text"].isnull().sum()
    + (df_reviews["Review Text"].str.strip() == "").sum()
)
print("Empty reviews:", empty_reviews_count)


EMPTY REVIEWS
Empty reviews: 0


0 empty/whitespace-only reviews — every row has real text to learn from.

## Class Balance - Counts

Logistic regression is a classifier: check how many reviews fall into each sentiment class. A skewed split (e.g. 90% positive) makes plain accuracy misleading, since always predicting the majority class would score high.

In [15]:
print(
    "Class counts (imbalance matters for accuracy):\n",
    df_reviews["sentiment"].value_counts(),
)

Class counts (imbalance matters for accuracy):
 sentiment
 1    17442
-1     2369
Name: count, dtype: int64


17,442 positive (1) vs 2,369 negative (-1) — the classes are heavily IMBALANCED (~88% / ~12%). A model that just predicts "positive" every time would score ~88% accuracy without learning anything — plan to check precision/recall/F1 (or use `class_weight="balanced"`) instead of relying on accuracy alone.

## Class Balance - Share (%)

Same check as above but as a percentage, easier to eyeball the imbalance ratio.

In [16]:
print("Class share")
print(df_reviews["sentiment"].value_counts(normalize=True).round(4))

Class share
sentiment
 1    0.8804
-1    0.1196
Name: proportion, dtype: float64


~88.04% positive vs ~11.96% negative — confirms the imbalance seen in the raw counts above, roughly a 7:1 ratio of positive to negative reviews.

## Unique Label Values

Confirm the exact label encoding (e.g. -1/1 vs 0/1) before training, since sklearn's classifiers treat labels as class identifiers, not numbers to do arithmetic on.

In [17]:
print("Label values:", sorted(df_reviews["sentiment"].unique().tolist()))

Label values: [-1, 1]


`[-1, 1]` — a clean binary label with no unexpected/stray values (e.g. no 0, no missing category), so no relabeling is needed.

## Feature and Target Selection

Same idea as the linear regression section: X = feature(s), y = target. The difference is that X here is raw TEXT, not numbers — it can't be fed to a model as-is, which is why the next step (vectorization) exists.

In [18]:
X = df_reviews["Review Text"]
y = df_reviews["sentiment"]

print(X.shape)
print(y.shape)

(19811,)
(19811,)


## Train-Test Split

**Split BEFORE vectorizing**, not after: fitting the vectorizer on the whole dataset (train + test combined) would leak information about the test set's vocabulary into the model. Splitting the raw text first keeps the test set genuinely unseen until prediction time. We also pass `stratify=y` since the classes are imbalanced (~88%/12%, see Class Balance checks above) — a plain random split could leave the test set with barely any negative reviews.

In [19]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set shape:", X_train_text.shape, y_train.shape)
print("Testing set shape:", X_test_text.shape, y_test.shape)

Training set shape: (15848,) (15848,)
Testing set shape: (3963,) (3963,)


15,848 training rows / 3,963 testing rows — the 80/20 split is preserved, just like the linear regression section.

## Text Vectorization (CountVectorizer)

Why vectorize: TV/Radio/Newspaper were already numbers, but "Review Text" is raw text -- models only understand numbersCountVectorizer (bag-of-words) turns each review into a vector of raw word-occurrence counts: every word in the vocabulary becomes a column, and each cell is how many times that word appears in that review. This step has no equivalent in the linear regression section because that data was already numeric.

**How sentences become columns (a tiny worked example):**

Say the training text was just these 3 reviews (assume `stop_words="english"` has already dropped `"i"`, `"this"`, `"is"`):

| Review | Kept words |
|---|---|
| "I love this dress" | love, dress |
| "I hate this dress" | hate, dress |
| "This dress is amazing" | dress, amazing |

`fit` scans all 3 reviews and builds the **vocabulary** — every distinct word becomes its own column, in alphabetical order: `amazing, dress, hate, love`. That's 4 columns for 3 reviews.

`transform` then counts, for each review, how many times each vocabulary word appears — turning text into a numeric table:

| Review | amazing | dress | hate | love |
|---|---|---|---|---|
| "I love this dress" | 0 | 1 | 0 | 1 |
| "I hate this dress" | 0 | 1 | 1 | 0 |
| "This dress is amazing" | 1 | 1 | 0 | 0 |

Each row is exactly what `LogisticRegression` receives as one training example — a row of numbers instead of a sentence.

**Why this explains `fit_transform` vs `transform`:** the 4 columns above (`amazing, dress, hate, love`) only exist because `fit` was run on these 3 reviews. If a brand-new review came in later — say `"I love this amazing dress"` — `transform` reuses those *same 4 columns* (`[1, 1, 0, 1]`); it does **not** add a 5th column for any word it hasn't seen before, and it never re-learns the vocabulary. That's exactly what the code below does at real scale: `fit_transform(X_train_text)` builds a (much bigger, 5,000-word) vocabulary from the training reviews only, and `transform(X_test_text)` reuses those same 5,000 columns for the test reviews.

Fit **only** on the training text, so the vocabulary is learned purely from training data — then just `transform()` the test text through that same vocabulary. Calling `fit_transform` on the test set (or on the full dataset before splitting) would leak test-set information into the model.

In [20]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words="english", max_features=5000)

X_train_bow = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print("Training feature matrix shape:", X_train_bow.shape)
print("Testing feature matrix shape:", X_test.shape)

Training feature matrix shape: (15848, 5000)
Testing feature matrix shape: (3963, 5000)


(15848, 5000) train / (3963, 5000) test — same 5,000-word vocabulary applied to both, but learned from the training text only.

In [21]:
print(X_train_bow[:5].toarray())  # Show the first 5 rows of the dense array representation

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


## Model Training

**LogisticRegression** instead of `LinearRegression`: it fits a decision boundary that outputs a class (-1 or 1) rather than a continuous number, by modeling the probability of each class with the logistic (sigmoid) function. `max_iter=1000` gives the optimizer enough iterations to converge with 5,000 features, and `solver="saga"` is a good choice for large datasets with many features, like our 5,000-word vocabulary.

In [22]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000, random_state=42, solver="saga")
log_model.fit(X_train_bow, y_train)

print("Model training complete! ✅")

print("Training accuracy:", log_model.score(X_train_bow, y_train))

print("Number of learned coefficients:", log_model.coef_.shape[1])
print("Intercept:", log_model.intercept_)

Model training complete! ✅
Training accuracy: 0.9741923271075215
Number of learned coefficients: 5000
Intercept: [1.12633423]


5,000 coefficients (one weight per word) — far too many to print individually like the 3 weights in the linear regression model, so we just report the count/intercept here.

## Overfitting Check (Train vs Test Accuracy)

Important distinction from the linear regression section: `.score()` on a classifier like `LogisticRegression` returns **accuracy** (% of correct predictions), not R², since R² doesn't apply to classification. The overfitting logic is the same though: compare train vs test.

In [23]:
train_accuracy = log_model.score(X_train_bow, y_train)
test_accuracy = log_model.score(X_test, y_test)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")

Training Accuracy: 0.9742
Testing Accuracy: 0.9288


Train 0.9742 vs Test 0.9288 — gap ~0.045. Still under the 0.05 rule-of-thumb, but noticeably wider than with TF-IDF weighting — raw word counts let the model fit the training vocabulary more tightly, pushing it closer to the overfitting boundary. Remember the imbalance warning from earlier too — accuracy alone can hide a model that's bad at the minority class, which the classification report below confirms.

## Classification Metrics (Confusion Matrix & Report)

Why accuracy isn't enough here: ~88% of reviews are positive, so a model could score ~88% accuracy by ignoring negative reviews entirely. The confusion matrix and classification report show performance **per class** (precision, recall, F1) instead of one overall number.

In [24]:
from sklearn.metrics import classification_report, confusion_matrix

test_predictions = log_model.predict(X_test)

section("CONFUSION MATRIX")
print(confusion_matrix(y_test, test_predictions))

section("CLASSIFICATION REPORT")
print(classification_report(y_test, test_predictions, digits=4))


CONFUSION MATRIX
[[ 291  183]
 [  99 3390]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

          -1     0.7462    0.6139    0.6736       474
           1     0.9488    0.9716    0.9601      3489

    accuracy                         0.9288      3963
   macro avg     0.8475    0.7928    0.8168      3963
weighted avg     0.9245    0.9288    0.9258      3963



Recall for the negative class (-1) is ~0.61 — the model still catches meaningfully fewer negative reviews than positive ones (recall 0.61 vs 0.97), even though overall accuracy looks strong at ~93%. This is the imbalance problem flagged earlier: accuracy alone hides how much worse the model does on the minority class.

## Prediction on New Data

New text must go through the **same** `vectorizer.transform()` (not `fit_transform`, which would relearn the vocabulary from this one sentence) before the model can use it — mirroring how the linear regression prediction needed a DataFrame shaped like the training features.

In [25]:
new_review = ["The quality of this dress is amazing, I love it!"]
new_review_vectorized = vectorizer.transform(new_review)
prediction = log_model.predict(new_review_vectorized)

print(f"Review: {new_review[0]}")
print(f"Predicted Sentiment: {prediction[0]}")

Review: The quality of this dress is amazing, I love it!
Predicted Sentiment: 1


Predicted Sentiment: 1 (positive) — matches the clearly positive wording.

## Precision, Recall & F1 Score (Manual Derivation)

Precision, recall, and F1 are all metrics that evaluate the performance of a classification model **per class** — unlike accuracy (one overall number), they show how well the model does on a single class specifically, which matters here since the classes are imbalanced.

- **Precision** = TP / (TP + FP) — of everything predicted positive, how much was actually positive? Low precision -> lots of false alarms.
- **Recall** = TP / (TP + FN) — of everything that IS actually positive, how much did the model correctly catch? Low recall -> the model misses a lot of actual positives.
- **F1 score** = 2 · (Precision · Recall) / (Precision + Recall) — the harmonic mean of the two; it only stays high when BOTH are high, so it can't be gamed by optimizing just one.

In [26]:
cm = confusion_matrix(y_test, test_predictions)
tn, fp, fn, tp = cm.ravel()  # labels sorted as [-1, 1] -> (class -1, class 1)

precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * (precision * recall) / (precision + recall)

section("MANUAL PRECISION / RECALL / F1 (positive class = 1)")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

section("CLASSIFICATION REPORT (sklearn)")
print(classification_report(y_test, test_predictions, digits=4))


MANUAL PRECISION / RECALL / F1 (positive class = 1)
Precision: 0.9488
Recall:    0.9716
F1 Score:  0.9601

CLASSIFICATION REPORT (sklearn)
              precision    recall  f1-score   support

          -1     0.7462    0.6139    0.6736       474
           1     0.9488    0.9716    0.9601      3489

    accuracy                         0.9288      3963
   macro avg     0.8475    0.7928    0.8168      3963
weighted avg     0.9245    0.9288    0.9258      3963



The `1` row of the classification report matches the manual precision/recall/F1 above exactly (0.9488 / 0.9716 / 0.9601), confirming the formulas.

### Note: Is this model "Optimistic" or "Pessimistic"?

Looking at the confusion matrix `[[291, 183], [99, 3390]]` (rows/cols ordered `[-1, 1]`):

- Of the 474 reviews that are **actually negative**, the model wrongly predicts **positive** for 183 of them (recall for class -1 is only 0.6139) -- it lets more than a third of real negative sentiment slip through as positive.
- Of the 3489 reviews that are **actually positive**, it only mislabels 99 as negative (recall for class 1 is 0.9716) -- it almost never mistakes a real positive for negative.

So the model's errors are lopsided in one direction: when it's unsure, it leans toward predicting **positive**. That makes this an **optimistic** model -- it gives the benefit of the doubt and assumes sentiment is good more often than it should, rather than a **pessimistic** model, which would err in the opposite direction (over-predicting the negative/unfavorable class and flagging too many genuine positives as negative).

This is a direct consequence of the class imbalance noted earlier (~88% positive vs ~12% negative, see Class Balance checks) -- the model saw far more positive examples during training, so it defaults to "positive" whenever the signal is ambiguous.


Confusion matrix for the negative class (-1) is the same matrix as above, just with the labels made explicit instead of relying on sklearn's default alphabetical/sorted order.

In [27]:
print("\nConfusion Matrix")
print(confusion_matrix(y_test, test_predictions, labels=[-1, 1]))


Confusion Matrix
[[ 291  183]
 [  99 3390]]
